In [1]:
# Imports

from pathlib import Path
from scipy.io import savemat
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re

In [2]:
# Define paths
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError: 
    BASE_DIR = Path.cwd().parent
    
INPUT_PATH = BASE_DIR / "analysis" / "participants"

print(BASE_DIR)
print(INPUT_PATH)

D:\thesis-matlab-pipeline
D:\thesis-matlab-pipeline\analysis\participants


In [3]:
# Get Participants
participants = [f.stem for f in list(INPUT_PATH.iterdir()) if f.is_dir()]
participants

['vp01',
 'vp02',
 'vp03',
 'vp04',
 'vp05',
 'vp06',
 'vp07',
 'vp08',
 'vp09_MH',
 'vp10_YM',
 'vp11_KL',
 'vp12_AJ',
 'vp13_HG',
 'vp14_UM',
 'vp15_LJ',
 'vp16_KR',
 'vp17_LM',
 'vp18_LS']

In [4]:
# Get logs in dir
def get_participant_logs(participant):
    participant_path = INPUT_PATH / participant / "logs"
    logs = [log.resolve() for log in participant_path.iterdir() if log.suffix == ".csv"]
    return logs

print(get_participant_logs(participants[0]))

[WindowsPath('D:/thesis-matlab-pipeline/analysis/participants/vp01/logs/JW1_corrected.csv'), WindowsPath('D:/thesis-matlab-pipeline/analysis/participants/vp01/logs/JW2_corrected.csv'), WindowsPath('D:/thesis-matlab-pipeline/analysis/participants/vp01/logs/JW3_corrected.csv')]


In [5]:
def build_arrays(log):
    log_conditions = set(log["condition"].dropna())
    
    if "coherent" in log_conditions:
        condition_order = ["coherent", "incoherent_real", "incoherent_mock", "Neutral_Real", "Neutral_Fake", "target"]
        
    elif "Congruent" in log_conditions:
        condition_order = ["Congruent", "Incongruent_Real", "Incongruent_Fake", "Neutral_Real", "Neutral_Fake", "True"]
        
    elif "Face" in log_conditions:
        condition_order = ["Face", "scene", "scrambled"]
        
    else:
        raise KeyError("Invalid log format!")
    
    columns = len(condition_order)

    onset_array = np.empty((columns,), dtype=np.object_)
    duration_array = np.empty((columns,), dtype=np.object_)
    names_array = np.empty((columns,), dtype=np.object_)

    return condition_order, onset_array, duration_array, names_array

In [6]:
def write_conditions_and_onsets(log, condition_order, onset_array):
    is_func = any(x in condition_order for x in ["Congruent", "coherent"])
    
    if is_func:
        onset_name = "_onset" if "Congruent" in condition_order else "_onset_cor"
        neutral_name = "Neutral" if "Congruent" in condition_order else "neutral"
        
    else:
        onset_name = "time when it onsets"
    
    for i, condition in enumerate(condition_order):
        if is_func:
            if condition.lower().startswith("neutral"):
                mask = (log["s2_category"].str.startswith("mock", na=False)
                        if condition.lower().endswith("_fake")
                        else ~log["s2_category"].str.startswith("mock", na=False))
            
                cond_onsets = log.loc[
                    (log["condition"] == neutral_name) & mask,
                    f"s1{onset_name}"].astype(float)

            else:
                cond_onsets = log.loc[
                    log["condition"] == condition, f"s1{onset_name}"
                    ].astype(float)
                
        else:
           cond_onsets = log.loc[
                log["condition"] == condition, onset_name
               ].astype(float)
            
           cond_onsets += 4

        onset_array[i] = cond_onsets.to_numpy()[:, None]

    return onset_array

In [7]:
def write_durations(duration_array, condition_order):
    duration_list = [0.20 for c in condition_order]

    for i, duration in enumerate(duration_list):
        duration_array[i] = duration

    return duration_array

In [8]:
def write_names(condition_order, names_array):

    if any(x in condition_order for x in ["Congruent", "coherent"]):
        condition_order = ["Congruent", "Incongruent_Real", "Incongruent_Fake", "Neutral_Real", "Neutral_Fake", "Target"] 
    
    for i, name in enumerate(condition_order):
        names_array[i] = name

    return names_array

In [9]:
def to_dict(onset_array, duration_array, names_array):
    mat_dict = {
        "onsets": onset_array,
        "names": names_array,
        "durations": duration_array,
    }

    return mat_dict

In [10]:
def prep_dir(log_id: str, participant: str):
    match = re.search(r"\d+", log_id)
    
    if match:
        log_number = match.group()
        func_dir = INPUT_PATH / participant / f"0{log_number}_func"
    else:
        func_dir = INPUT_PATH / participant / "00_loc"
    
    func_dir.mkdir(parents=True, exist_ok=True)

    return func_dir

In [11]:
def save_mcf(output_dir: Path, mat_dict: dict):
    out_dir = output_dir / "mcf"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_file = out_dir / "MCF_cue_split.mat"
    savemat(out_file, mat_dict)

In [12]:
# Run conversion for all files
def log_to_mcf_conversion(logs, participant):
    for log in logs:
        log_path = INPUT_PATH / log
        df = pd.read_csv(log_path)
    
        condition_order, empty_onset_array, empty_duration_array, empty_name_array = build_arrays(df)
        
        onset_array = write_conditions_and_onsets(df, condition_order, empty_onset_array)
        duration_array = write_durations(empty_duration_array, condition_order)
        names_array = write_names(condition_order, empty_name_array)
        
        mat_dict = to_dict(onset_array, duration_array, names_array)
        
        mcf_dir = prep_dir(log.stem, participant)
        save_mcf(mcf_dir, mat_dict)
        
        print(f"MCF for {log} saved to {mcf_dir}.")

In [13]:
for participant in participants:
    logs = get_participant_logs(participant)
    log_to_mcf_conversion(logs, participant)
    print(f"MCF conversion completed for {logs}.")

MCF for D:\thesis-matlab-pipeline\analysis\participants\vp01\logs\JW1_corrected.csv saved to D:\thesis-matlab-pipeline\analysis\participants\vp01\01_func.
MCF for D:\thesis-matlab-pipeline\analysis\participants\vp01\logs\JW2_corrected.csv saved to D:\thesis-matlab-pipeline\analysis\participants\vp01\02_func.
MCF for D:\thesis-matlab-pipeline\analysis\participants\vp01\logs\JW3_corrected.csv saved to D:\thesis-matlab-pipeline\analysis\participants\vp01\03_func.
MCF conversion completed for [WindowsPath('D:/thesis-matlab-pipeline/analysis/participants/vp01/logs/JW1_corrected.csv'), WindowsPath('D:/thesis-matlab-pipeline/analysis/participants/vp01/logs/JW2_corrected.csv'), WindowsPath('D:/thesis-matlab-pipeline/analysis/participants/vp01/logs/JW3_corrected.csv')].
MCF for D:\thesis-matlab-pipeline\analysis\participants\vp02\logs\SG1_corrected.csv saved to D:\thesis-matlab-pipeline\analysis\participants\vp02\01_func.
MCF for D:\thesis-matlab-pipeline\analysis\participants\vp02\logs\SG2_cor